In [4]:
from selenium import webdriver
from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from bs4 import BeautifulSoup
import csv  # 导入 csv 模块

# 使用 Chrome 瀏覽器
driver = webdriver.Chrome()

# 打開目標網頁
driver.get("https://map.ctop.tw/ctopmap")  # 替換為實際的目標網址

# 等待帳號輸入框加載
account_input = WebDriverWait(driver, 5).until(
    EC.presence_of_element_located((By.ID, "account"))
)

# 輸入帳號
account_input.send_keys("0900612080")  # 替換為你的帳號

# 輸入密碼
passwd_input = driver.find_element(By.ID, "passwd")
passwd_input.send_keys("benson777")  # 替換為你的密碼

# 找到並點擊登入按鈕
login_button = driver.find_element(By.ID, "btn_login")
login_button.click()

# 等待登入成功並跳轉到查詢頁面（這裡設置 10 秒等待）
WebDriverWait(driver, 10).until(
    EC.url_changes("https://ctopmap.ctop.tw/home")  # 替換成目標頁面 URL
)

for i in range(1, 5):
    # 動態生成每個按鈕的 class 名稱
    btn_class = f"btnS{i}"
    
    # 等待按鈕可點擊，並找到對應的按鈕
    span_button = WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, f"span.{btn_class}"))
    )
    
    # 點擊“我知道了”按鈕
    span_button.click()

# 打開 CSV 文件，準備寫入數據
with open('facilities_data.csv', 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['city', 'district', 'facility_name', 'x_coordinate', 'y_coordinate']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    
    # 寫入 CSV 表頭
    writer.writeheader()

    # 找到縣市的下拉選單元素
    select_city_element = driver.find_element(By.ID, "selAdvCity")
    select_city = Select(select_city_element)

    # 取得所有縣市的選項
    city_options = select_city.options

    for i in range(1, len(city_options)):
        # 選擇當前的縣市
        select_city.select_by_index(i)
        
        # 取得當前選中的縣市名稱
        city_name = city_options[i].text
        print(f"選擇縣市: {city_name}")

        # 等待第二個下拉選單的地區選項加載完成
        select_dist_element = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.ID, "selAdvDist"))
        )
        
        # 初始化地區下拉選單
        select_dist = Select(select_dist_element)
        
        # 取得所有區域的選項
        dist_options = select_dist.options
        
        # 遍歷每個區
        for j in range(1, len(dist_options)):
            dist_options = select_dist.options

            # 選擇當前的區
            select_dist.select_by_index(j)
                
            # 取得當前選中的區名稱
            dist_name = dist_options[j].text
            print(f"    選擇地區: {dist_name}")

            # 等待并确保搜索按钮可见和可点击
            search_button = WebDriverWait(driver, 10).until(
                EC.visibility_of_element_located((By.CSS_SELECTOR, "a.search-button.icon.search.active"))
            )

            search_button.click()
            time.sleep(3)
                
            back_button = WebDriverWait(driver, 10).until(
                EC.visibility_of_element_located((By.ID, "toggle_aside_nav_status_red_bar"))
            )
            back_button.click()
                
            # 等待嫌惡設施的隱藏字段加載完成
            hidden_input_element = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.ID, "disgusting"))
            )
                
            hidden_value = hidden_input_element.get_attribute("value")
                
            # 解析HTML並提取設施名稱和座標
            soup = BeautifulSoup(hidden_value, 'html.parser')
            facilities = []
            # 遍歷每個含有設施資料的tr標籤
            for row in soup.find_all('tr', class_='analy_content_2'):
                # 提取設施名稱
                facility_name = row.find_all('td')[1].text
                    
                # 提取坐標信息（在onclick屬性中）
                onclick_text = row.find('img')['onclick']
                    
                # 使用字符串分割提取出經緯度
                parts = onclick_text.split(',')
                x_coordinate = parts[2]
                y_coordinate = parts[3].split("'")[0]  # 去除末尾的引號
                    
                # 将提取的数据写入 CSV 文件
                writer.writerow({
                    'city': city_name,
                    'district': dist_name,
                    'facility_name': facility_name,
                    'x_coordinate': x_coordinate,
                    'y_coordinate': y_coordinate
                })
                
                # 打印提取的設施名稱和座標
                print(f"    名稱: {facility_name}, X座標: {x_coordinate}, Y座標: {y_coordinate}")
                
        # 在選擇下一個縣市之前，可以視情況等待一段時間
        time.sleep(1)

# 爬取完所有縣市和區後關閉瀏覽器
driver.quit()


選擇縣市: 基隆市
    選擇地區: 仁愛區
    名稱: 三友煤氣行, X座標: 324931.53800000035, Y座標: 2779659.7270000004
    名稱: 三興宮, X座標: 324300.22831093456, Y座標: 2779527.85396078
    名稱: 天王宮, X座標: 325789.98804786976, Y座標: 2779772.28398274
    名稱: 天澤宮, X座標: 325549.62509372307, Y座標: 2779412.40001911
    名稱: 天濟宮, X座標: 325942.2030976572, Y座標: 2779794.89483232
    名稱: 太乙宮, X座標: 324932.5840347415, Y座標: 2780104.95400611
    名稱: 木山殯儀有限公司, X座標: 324341.38099999994, Y座標: 2779981.453
    名稱: 代明宮, X座標: 324558.35915299226, Y座標: 2780301.29252119
    名稱: 古今葬儀社, X座標: 324890.4390000012, Y座標: 2779832.4919999996
    名稱: 台灣中油成功一路站加油站, X座標: 324373.06900000025, Y座標: 2780029.01
    名稱: 台灣中油第一流動站加油站, X座標: 324353.89099999925, Y座標: 2780061.713
    名稱: 台灣國寶殯葬禮儀有限公司, X座標: 325873.1289999997, Y座標: 2780144.808
    名稱: 弘道禮儀企業社, X座標: 325732.3039999991, Y座標: 2779929.117
    名稱: 永生禮儀用品社, X座標: 324987.54399999965, Y座標: 2778854.109
    名稱: 永玖禮儀用品店, X座標: 324652.65900000033, Y座標: 2780008.578
    名稱: 永龍國際禮儀有限公司, X座標: 324634.6469999998, Y座標: 2778511.865
    

KeyboardInterrupt: 